## Foundry Agent with Code Interpreter Capability

![lab_flow](./Assets/lab_flow.png)

### Installing Required Libraries

In [1]:

%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\zvijayfa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Setting Up the Environment Variables

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool, CodeInterpreterToolAuto

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

### Setting Up the Foundry Project Client

In [3]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Creating the OpenAI Client

In [4]:
openai_client = project_client.get_openai_client()

### CSV File Upload

In [5]:
# Upload the CSV file for the code interpreter to use
file = openai_client.files.create(purpose="assistants", file=open("./electronics_products.csv", "rb"))
print(f"File uploaded (id: {file.id})")

File uploaded (id: assistant-EQpPzeruZm82v7xi3729Gs)


### Creating Agent with the Code Interpreter Tool

In [6]:
agent_name = "code-interpreter-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful AI Assistant with code interpreter capabilities.",
        tools = [
            CodeInterpreterTool(
                container = CodeInterpreterToolAuto(
                    file_ids = [file.id]
                )
            )
        ]
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: code-interpreter-agent:1, name: code-interpreter-agent, version: 1)


### Creating a Conversation Object for the Agent Chat System

In [7]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_82b1eb6129b46f3300Jpf9HPghAKTCMxRRJHmeXZbwQFmhk8qy


### Calling Our Agent

In [8]:
response = openai_client.responses.create(
    conversation = conversation.id,
    input = "Could you please create a column chart with products on the x-axis and their respective prices on the y-axis?",
    extra_body = {
        "agent": {
            "name": agent.name,
            "type": "agent_reference"
        }
    }
)

print(f"Response completed with id: {response.id}")

Response completed with id: resp_82b1eb6129b46f330069ede13a144c8190ad6553d5a8458985


### Extracting file information from response annotations

In [9]:
file_id = ""
filename = ""
container_id = ""

# Get the last message which should contain file citations
last_message = response.output[-1]  # ResponseOutputMessage
if last_message.type == "message":
        # Get the last content item (contains the file annotations)
        text_content = last_message.content[-1]  # ResponseOutputText
        if text_content.type == "output_text":
            # Get the last annotation (most recent file)
            if text_content.annotations:
                file_citation = text_content.annotations[-1]  # AnnotationContainerFileCitation
                if file_citation.type == "container_file_citation":
                    file_id = file_citation.file_id
                    filename = file_citation.filename
                    container_id = file_citation.container_id
                    print(f"Found generated file: {filename} (ID: {file_id})")

# Download the generated file if available
if file_id and filename:
        file_content = openai_client.containers.files.content.retrieve(file_id=file_id, container_id=container_id)
        with open(filename, "wb") as f:
            f.write(file_content.read())
            print(f"File {filename} downloaded successfully.")
        print(f"File ready for download: {filename}")
else:
        print("No file generated in response")

Found generated file: products_price_chart.png (ID: cfile_69ede1471854819098fb560ebb7261e2)
File products_price_chart.png downloaded successfully.
File ready for download: products_price_chart.png
